# NeuroVLM PubMed Recall Baseline

Local notebook for checking the pretrained NeuroVLM contrastive brain-to-text baseline on PubMed only. The main question is: given a PubMed brain image/latent from a paper, does the model rank that same paper high among PubMed text candidates?

This uses the no-generation `NeuroVLM.to_text(...)` path in `core.py`, restricts candidates to the project-provided PubMed `test` split from the publications dataframe, and reports recall@1/5/10/50, MRR, median rank, random baselines, and the paper-style full recall-curve AUC.


## 1. Imports / Device

In [10]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

repo_root = Path.cwd()
if (repo_root / 'src').exists():
    sys.path.insert(0, str(repo_root / 'src'))

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu')
print('device:', device)
print('torch:', torch.__version__)


device: mps
torch: 2.10.0


## 2. Config

In [11]:
SPLIT_COLUMN = "test"
BATCH_SIZE = 4096
OUT_DIR = Path('runs/neurovlm_pubmed_recall_baseline')
OUT_DIR.mkdir(parents=True, exist_ok=True)
KS = (1, 5, 10, 50)
print(OUT_DIR.resolve())


/Users/borng/code/lab_work/neurovlm/experiments/runs/neurovlm_pubmed_recall_baseline


## 3. Load PubMed Latents And Projection Heads

In [12]:
from neurovlm.core import NeuroVLM
from neurovlm.retrieval_resources import _load_latent_neuro, _load_latent_text
from neurovlm.data import load_dataset

# Use the project retrieval resources for saved Hugging Face latents.
# The actual brain-to-text scoring below uses NeuroVLM.to_text(...), i.e.
# the core no-generation retrieval path.
brain_latent, brain_pmids = _load_latent_neuro()
text_latent, text_pmids = _load_latent_text()
pub_df = load_dataset("pubmed_text").copy()

brain_pmids = np.asarray(brain_pmids).astype(str)
text_pmids = np.asarray(text_pmids).astype(str)
pub_df["pmid"] = pub_df["pmid"].astype(str)

nvlm = NeuroVLM(datasets=["pubmed"], device=str(device))

print('brain_latent:', tuple(brain_latent.shape), 'brain_pmids:', len(brain_pmids))
print('text_latent :', tuple(text_latent.shape), 'text_pmids :', len(text_pmids))
print('publications:', pub_df.shape, 'columns:', list(pub_df.columns))
print('NeuroVLM datasets:', nvlm.datasets)


brain_latent: (29868, 384) brain_pmids: 29868
text_latent : (30826, 768) text_pmids : 30826
publications: (30826, 8) columns: ['pmid', 'pmcid', 'doi', 'name', 'description', 'train', 'test', 'val']
NeuroVLM datasets: ('pubmed',)


## 4. Align By PMID

In [13]:
# Alignment follows text PMID order, matching the ALE dataset logic.
# After this, aligned_brain[i] and aligned_text[i] are the true paired paper examples,
# so diagonal entries in the retrieval similarity matrix are the correct matches.
brain_lookup = {p: i for i, p in enumerate(brain_pmids)}
brain_rows, text_rows, shared_pmids = [], [], []
for text_i, pmid in enumerate(text_pmids):
    brain_i = brain_lookup.get(pmid)
    if brain_i is None:
        continue
    brain_rows.append(brain_i)
    text_rows.append(text_i)
    shared_pmids.append(pmid)

brain_rows = np.asarray(brain_rows, dtype=np.int64)
text_rows = np.asarray(text_rows, dtype=np.int64)
shared_pmids = np.asarray(shared_pmids).astype(str)
aligned_brain_pmids = brain_pmids[brain_rows]
aligned_text_pmids = text_pmids[text_rows]
assert np.array_equal(aligned_brain_pmids, aligned_text_pmids)

print('aligned pairs:', len(shared_pmids))
print('first 5 alignment checks:')
for i in range(5):
    print(i, 'pmid=', shared_pmids[i], 'brain_pmid=', aligned_brain_pmids[i], 'text_pmid=', aligned_text_pmids[i])


aligned pairs: 29868
first 5 alignment checks:
0 pmid= 1589767 brain_pmid= 1589767 text_pmid= 1589767
1 pmid= 8530552 brain_pmid= 8530552 text_pmid= 8530552
2 pmid= 8624678 brain_pmid= 8624678 text_pmid= 8624678
3 pmid= 8670634 brain_pmid= 8670634 text_pmid= 8670634
4 pmid= 8994101 brain_pmid= 8994101 text_pmid= 8994101


## 5. Use Existing PubMed Test Split

In [14]:
# Use the project-provided split column in the publications dataframe.
# This avoids comparing ALE experiments against a different random split.
if SPLIT_COLUMN not in pub_df.columns:
    raise KeyError(f"Missing split column {SPLIT_COLUMN!r}; available columns: {list(pub_df.columns)}")

split_lookup = pub_df.set_index("pmid")[SPLIT_COLUMN]

def as_bool_split_value(x):
    if pd.isna(x):
        return False
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    if isinstance(x, (int, np.integer, float, np.floating)):
        return bool(int(x))
    return str(x).strip().lower() in {"1", "true", "t", "yes", "y", "test"}

test_mask = np.asarray([as_bool_split_value(split_lookup.get(pmid, False)) for pmid in shared_pmids])
test_pos = np.flatnonzero(test_mask)
if len(test_pos) == 0:
    raise RuntimeError(f"No aligned PMIDs are marked as {SPLIT_COLUMN}=True/test.")

print('split column:', SPLIT_COLUMN)
print('aligned pairs:', len(shared_pmids))
print('test pairs:', len(test_pos))
print('test fraction:', len(test_pos) / len(shared_pmids))


split column: test
aligned pairs: 29868
test pairs: 2987
test fraction: 0.10000669612963707


## 6. Score Test Brain Latents With Core `to_text`

In [15]:
# Query payload: saved PubMed brain latents for the project test split.
# NeuroVLM.to_text(..., project=True) projects these 384-d brain latents with
# the pretrained image_infonce projection head and scores against projected
# PubMed SPECTER text embeddings. No generation is used.
test_brain_raw = brain_latent[brain_rows[test_pos]].float()
test_pmids = shared_pmids[test_pos]
candidate_text_rows = text_rows[test_pos]
candidate_pmids = text_pmids[candidate_text_rows]
assert np.array_equal(test_pmids, candidate_pmids)

result = nvlm.to_text(test_brain_raw, datasets=["pubmed"], project=True)
scores_all = result.scores_by_dataset["pubmed"]  # shape: all PubMed text candidates x test brain queries
sim = scores_all[candidate_text_rows].T.contiguous()  # shape: test queries x test candidates

print('retrieval_space:', result.retrieval_space)
print('all PubMed score matrix:', tuple(scores_all.shape))
print('test-only brain-to-paper sim:', tuple(sim.shape))
print('diagonal PMIDs are true pairs:', np.array_equal(test_pmids, candidate_pmids))


retrieval_space: shared
all PubMed score matrix: (30826, 2987)
test-only brain-to-paper sim: (2987, 2987)
diagonal PMIDs are true pairs: True


## 7. Brain Image -> Paper Retrieval Metrics

In [16]:
from neurovlm.metrics import retrieval_ranks

def metrics_from_similarity(sim, ks=KS):
    ranks = retrieval_ranks(sim).float()
    n = float(sim.shape[0])
    out = {
        'median_rank': float(ranks.median().item()),
        'mean_rank': float(ranks.mean().item()),
        'mrr': float((1.0 / ranks).mean().item()),
    }
    for k in ks:
        out[f'recall@{k}'] = float((ranks <= k).float().mean().item())
        out[f'random_recall@{k}'] = min(float(k) / n, 1.0)
    counts = torch.bincount((ranks.long() - 1), minlength=sim.shape[0])
    curve = counts.cumsum(0).float() / float(sim.shape[0])
    out['paper_recall_curve_auc'] = float(curve.mean().item())
    out['recall_curve'] = curve
    return out

# Main requested direction: query = PubMed brain image/latent, target = PubMed paper text.
# Correct paper is on the diagonal because candidate_text_rows uses the same test PMIDs/order.
brain_to_paper = metrics_from_similarity(sim, ks=KS)
paper_to_brain = metrics_from_similarity(sim.T, ks=KS)

summary = {
    'n_test': len(test_pmids),
    'brain_to_paper_paper_recall_curve_auc': brain_to_paper['paper_recall_curve_auc'],
    'brain_to_paper_recall@1': brain_to_paper['recall@1'],
    'brain_to_paper_recall@5': brain_to_paper['recall@5'],
    'brain_to_paper_recall@10': brain_to_paper['recall@10'],
    'brain_to_paper_recall@50': brain_to_paper['recall@50'],
    'brain_to_paper_mrr': brain_to_paper['mrr'],
    'brain_to_paper_median_rank': brain_to_paper['median_rank'],
    'brain_to_paper_random_recall@10': brain_to_paper['random_recall@10'],
    'paper_to_brain_paper_recall_curve_auc': paper_to_brain['paper_recall_curve_auc'],
    'paper_to_brain_recall@10': paper_to_brain['recall@10'],
    'mean_paper_recall_curve_auc': (brain_to_paper['paper_recall_curve_auc'] + paper_to_brain['paper_recall_curve_auc']) / 2.0,
    'mean_recall@10': (brain_to_paper['recall@10'] + paper_to_brain['recall@10']) / 2.0,
}
summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})
summary_df


,value
n_test,2987.000000
brain_to_paper_paper_recall_curve_auc,0.833709
brain_to_paper_recall@1,0.010378
brain_to_paper_recall@5,0.051892
brain_to_paper_recall@10,0.093070
brain_to_paper_recall@50,0.241379
brain_to_paper_mrr,0.039668
brain_to_paper_median_rank,217.000000
brain_to_paper_random_recall@10,0.003348
paper_to_brain_paper_recall_curve_auc,0.836605


## 8. Save Results And Recall Curves

In [17]:
recall_curves = {
    'k_values': list(range(1, len(test_pmids) + 1)),
    'brain_to_paper_recall_curve': brain_to_paper['recall_curve'].cpu().tolist(),
    'paper_to_brain_recall_curve': paper_to_brain['recall_curve'].cpu().tolist(),
    'brain_to_paper_auc': brain_to_paper['paper_recall_curve_auc'],
    'paper_to_brain_auc': paper_to_brain['paper_recall_curve_auc'],
    'random_recall_curve': (torch.arange(1, len(test_pmids) + 1).float() / float(len(test_pmids))).tolist(),
}

payload = {
    'config': {'split_column': SPLIT_COLUMN, 'split_source': 'pubmed_text publications dataframe', 'candidate_set': 'pubmed test split only'},
    'summary': summary,
    'brain_to_paper_metrics': {k: v for k, v in brain_to_paper.items() if k != 'recall_curve'},
    'paper_to_brain_metrics': {k: v for k, v in paper_to_brain.items() if k != 'recall_curve'},
    'recall_curves': recall_curves,
}
with open(OUT_DIR / 'pubmed_test_recall_metrics.json', 'w') as f:
    json.dump(payload, f, indent=2)
summary_df.to_csv(OUT_DIR / 'pubmed_test_recall_summary.csv')
pd.DataFrame(recall_curves).to_csv(OUT_DIR / 'pubmed_test_recall_curves.csv', index=False)
print('saved:', OUT_DIR.resolve())


saved: /Users/borng/code/lab_work/neurovlm/experiments/runs/neurovlm_pubmed_recall_baseline


## 9. Inspect Worst / Best Brain-to-Paper Ranks

In [18]:
ranks = retrieval_ranks(sim).cpu().numpy()
rank_df = pd.DataFrame({
    'pmid': test_pmids,
    'rank': ranks,
    'hit@10': ranks <= 10,
    'true_score': sim.diag().cpu().numpy(),
    'top1_pos': sim.argmax(dim=1).cpu().numpy(),
    'top1_score': sim.max(dim=1).values.cpu().numpy(),
})
rank_df['top1_pmid'] = test_pmids[rank_df['top1_pos'].to_numpy()]
rank_df.to_csv(OUT_DIR / 'pubmed_test_brain_to_paper_ranks.csv', index=False)
display(rank_df.sort_values('rank').head(10))
display(rank_df.sort_values('rank', ascending=False).head(10))


,pmid,rank,hit@10,true_score,top1_pos,top1_score,top1_pmid
1381,25061565,1,True,0.358555,1381,0.358555,25061565
2482,33414733,1,True,0.411015,2482,0.411015,33414733
598,20204074,1,True,0.388027,598,0.388027,20204074
2176,30798012,1,True,0.353785,2176,0.353785,30798012
271,17126370,1,True,0.405632,271,0.405632,17126370
469,19240659,1,True,0.364428,469,0.364428,19240659
1014,23046904,1,True,0.410907,1014,0.410907,23046904
2445,33192481,1,True,0.350599,2445,0.350599,33192481
23,11230097,1,True,0.395947,23,0.395947,11230097
1006,22993433,1,True,0.421701,1006,0.421701,22993433


,pmid,rank,hit@10,true_score,top1_pos,top1_score,top1_pmid
1936,28782545,2968,False,-0.202803,486,0.385454,19400679
2953,38342187,2955,False,-0.198220,1933,0.417467,28760866
76,13010008,2921,False,-0.205250,720,0.487716,21289179
2171,30735675,2913,False,-0.072319,2402,0.318916,32664021
92,14704218,2870,False,-0.053094,2908,0.319408,37743995
123,15294149,2856,False,-0.085480,525,0.337610,19646537
1694,26961091,2855,False,-0.083369,1736,0.431670,27262510
2839,36914701,2845,False,-0.205786,789,0.368377,21718759
49,12457757,2840,False,-0.146615,2257,0.408359,31439831
2029,29501856,2835,False,-0.118928,1263,0.341592,24375687


## 10. Single PMID Sanity Check

In [19]:
# Pick one PMID from the PubMed test split and inspect its brain-to-paper ranking.
# Change SINGLE_PMID to any PMID in test_pmids, or leave it as None to use the first test item.
SINGLE_PMID = None
TOP_K = 20

if SINGLE_PMID is None:
    query_pos = 0
else:
    matches = np.flatnonzero(test_pmids == str(SINGLE_PMID))
    if len(matches) == 0:
        raise ValueError(f"PMID {SINGLE_PMID} is not in the PubMed test split.")
    query_pos = int(matches[0])

query_pmid = str(test_pmids[query_pos])
query_brain = brain_latent[brain_rows[test_pos[query_pos]]].float().unsqueeze(0)
single_result = nvlm.to_text(query_brain, datasets=["pubmed"], project=True)
single_scores_all = single_result.scores_by_dataset["pubmed"][:, 0]
single_scores_test = single_scores_all[candidate_text_rows]
order = torch.argsort(single_scores_test, descending=True)
true_rank = int((order == query_pos).nonzero(as_tuple=True)[0].item()) + 1

true_row = pub_df.loc[pub_df["pmid"] == query_pmid].head(1).copy()
print("Query/test PMID:", query_pmid)
print("True paper rank among PubMed test candidates:", true_rank, "of", len(test_pmids))
print("True paper metadata:")
display(true_row.T)

top_positions = order[:TOP_K].cpu().numpy()
top_pmids = test_pmids[top_positions]
top_scores = single_scores_test[top_positions].cpu().numpy()
top_df = pd.DataFrame({
    "rank": np.arange(1, len(top_positions) + 1),
    "pmid": top_pmids,
    "score": top_scores,
    "is_true_paper": top_pmids == query_pmid,
})
meta_cols = [c for c in ["pmid", "title", "abstract", "description"] if c in pub_df.columns]
top_df = top_df.merge(pub_df[meta_cols], on="pmid", how="left")
display(top_df)


Query/test PMID: 8994101
True paper rank among PubMed test candidates: 22 of 2987
True paper metadata:


,26865
pmid,8994101
pmcid,NaN
doi,None
name,Acoustic neuroma: correlations between morphol...
description,Forty-two patients with acoustic neuroma (AN) ...
train,False
test,True
val,False


,rank,pmid,score,is_true_paper,description
0,1,21329758,0.520293,False,This paper presents a framework for creating n...
1,2,19349231,0.495528,False,Functional MRI studies on humans with BOLD con...
2,3,18761038,0.483327,False,Simultaneous recording of electrophysiology an...
3,4,19778619,0.463279,False,Previous studies using combined electrical and...
4,5,31470126,0.461352,False,The complex transverse water proton magnetizat...
5,6,34964194,0.450162,False,MRI has a crucial role in presurgical evaluati...
6,7,22328419,0.434059,False,"In this work, we investigate the feasibility t..."
7,8,18515149,0.422854,False,We describe a Bayesian inference scheme for qu...
8,9,28559192,0.421606,False,In arterial spin labeling (ASL) a perfusion we...
9,10,27693612,0.419230,False,"We propose BrainNetCNN, a convolutional neural..."
